### Crie uma tabela de Clientes para equipe de Marketing
Importante -> Somente pessoas que possuam e-mail e telefone cadastrados

In [0]:
%python
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
%python
# Criando várias tabelas temporárias 
bronze_map = {
    #'tmp_bronze_brands': f'{bronze_path}/brands/',
    #'tmp_bronze_categories': f'{bronze_path}/categories/',
    'tmp_bronze_customers': f'{bronze_path}/customers/',
    #'tmp_bronze_order_items': f'{bronze_path}/order_items/',
    #'tmp_bronze_orders': f'{bronze_path}/orders/',
    #'tmp_bronze_products': f'{bronze_path}/products/',
    #'tmp_bronze_staffs': f'{bronze_path}/staffs/',
    #'tmp_bronze_stocks': f'{bronze_path}/stocks/',
    #'tmp_bronze_stores': f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
SELECT
    CT.customer_id,
    CT.first_name,
    CT.last_name,
    CT.phone,
    CT.email,
    CT.street,
    CT.city,
    CT.state,
    CT.zip_code
FROM tmp_bronze_customers AS CT
WHERE 1 = 1
AND CT.phone IS NOT NULL -- Vazio
AND CT.phone NOT IN ('NULL', 'NULL ') -- Texto NULL
AND CT.email IS NOT NULL -- Vazio
AND CT.email not in ('NULL', 'NULL ') -- Texto NULL

In [0]:
DESCRIBE tmp_bronze_customers

In [0]:
%python
## Salvando DataFrame na camada Silver
df_customer_silver = spark.sql("""
-- Consulta de clientes
SELECT
    CT.customer_id,
    CT.first_name,
    CT.last_name,
    CT.phone,
    CT.email,
    CT.street,
    CT.city,
    CT.state,
    CT.zip_code
FROM tmp_bronze_customers AS CT
WHERE 1 = 1
AND CT.phone IS NOT NULL -- Vazio
AND CT.phone NOT IN ('NULL', 'NULL ') -- Texto NULL
AND CT.email IS NOT NULL -- Vazio
AND CT.email not in ('NULL', 'NULL ') -- Texto NULL
""")

# Salvar e transformar em formato delta na camada Silver
df_customer_silver.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{silver_path}/customer')

In [0]:
/*Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_customer
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/customer';

In [0]:
SELECT * FROM bikestore.logistics.silver_customer LIMIT 15